# Week 3 Exercise: Synthetic Q&A Dataset Generator

The Week 3 challenge is: use a variety of models and prompts to generate a diverse synthetic dataset,
with a Gradio UI, for a real business use case.

This ties back to the technical tutor built in the [Week 1](../../week1/week1%20EXERCISE.ipynb) and
[Week 2](../../week2/community-contributions/technical_tutor_pro_waruts.ipynb) exercises: instead of a
generic "fake customer records" generator, this generates a **question/answer training dataset for a
technical tutor** - genuinely useful for evaluating or fine-tuning it later in the course.

Three models play three different roles, so "diverse outputs from a variety of models" actually means
something rather than just picking a random model per row:

- One model **writes the questions**
- A different model **writes the answers** (so it isn't grading its own homework)
- A third model acts as a **judge**, scoring each pair 1-5 and filtering out weak ones before they land
  in the final dataset

Set `RUN_LOCAL_MODEL = False` below if you don't have Ollama running, and leave `ANTHROPIC_API_KEY`
unset if you don't want Claude - both are detected automatically.

In [1]:
# imports

import os
import re
import json
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import gradio as gr

## Setting up

Same pattern as Week 1 and Week 2: load keys from `.env`, only enable a model if we can actually reach it.

In [2]:
# set up environment

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

RUN_LOCAL_MODEL = True  # set False to skip Ollama entirely
OLLAMA_BASE_URL = "http://localhost:11434/v1"
LOCAL_MAX_TOKENS = 300  # local CPU models can be slow - keep answers bounded, as in the Week 1/2 exercises

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set - the Claude option will be hidden")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [3]:
# constants and clients - only create the ones we actually have credentials/access for

MODEL_GPT = "gpt-4o-mini"
MODEL_CLAUDE = "claude-sonnet-4-5-20250929"
MODEL_OLLAMA = "qwen3.5:latest"  # swap for whatever you have pulled, e.g. llama3.2

openai_client = OpenAI() if openai_api_key else None

claude_client = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/"
) if anthropic_api_key else None

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama") if RUN_LOCAL_MODEL else None

# label -> (client, model name)
MODELS = {}
if openai_client:
    MODELS["GPT-4o-mini"] = (openai_client, MODEL_GPT)
if claude_client:
    MODELS["Claude Sonnet"] = (claude_client, MODEL_CLAUDE)
if ollama_client:
    MODELS[f"Ollama ({MODEL_OLLAMA}, local)"] = (ollama_client, MODEL_OLLAMA)

if len(MODELS) < 2:
    raise RuntimeError("Need at least 2 models available (e.g. an API key plus Ollama) to fill the three roles usefully")

print("Available models:", list(MODELS.keys()))

Available models: ['GPT-4o-mini', 'Claude Sonnet', 'Ollama (qwen3.5:latest, local)']


## The three roles

One `chat_once` helper, reused for all three roles (question-writer, answer-writer, judge) - they only differ in system prompt and how the response gets parsed.

In [4]:
def chat_once(model_choice, system_prompt, user_prompt):
    client, model = MODELS[model_choice]
    kwargs = {"max_tokens": LOCAL_MAX_TOKENS} if model == MODEL_OLLAMA else {}
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        **kwargs
    )
    return response.choices[0].message.content


def extract_json(text):
    # models sometimes wrap JSON in markdown fences or add a sentence of preamble -
    # grab the first top-level [...] or {...} block and parse that
    match = re.search(r"(\[.*\]|\{.*\})", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in model output: {text!r}")
    return json.loads(match.group(1))

In [5]:
def generate_questions(model_choice, topic, difficulty, count):
    system_prompt = (
        "You are an expert question-setter building a technical quiz dataset. "
        "Respond with ONLY a JSON array of strings - no other text, no markdown fences."
    )
    user_prompt = (
        f"Write {count} distinct, well-formed technical questions about '{topic}', "
        f"suitable for a {difficulty.lower()} learner. Vary the phrasing and angle of each question so "
        f"they don't feel repetitive. Return them as a JSON array of strings."
    )
    questions = extract_json(chat_once(model_choice, system_prompt, user_prompt))
    return [str(q).strip() for q in questions][:count]

In [6]:
def generate_answer(model_choice, question, difficulty):
    system_prompt = (
        f"You are a precise technical tutor. Answer the learner's question clearly and correctly, "
        f"pitched at a {difficulty.lower()} level. Use markdown, with short code examples in code "
        f"blocks where they would help."
    )
    return chat_once(model_choice, system_prompt, question).strip()

In [7]:
def judge_pair(model_choice, question, answer, difficulty):
    system_prompt = (
        "You are a strict technical reviewer grading a question-answer pair for a training dataset. "
        "Respond with ONLY a JSON object: {\"score\": <integer 1-5>, \"notes\": \"<one short sentence>\", "
        "\"assessed_difficulty\": \"<Beginner, Intermediate, or Advanced>\"}. "
        "5 = accurate, clear and complete. 1 = wrong or unusable. "
        "assessed_difficulty is your own honest judgement of the level this pair actually reads at - "
        "it does not need to match the level it was written for."
    )
    user_prompt = f"Question: {question}\n\nAnswer: {answer}\n\n(Written for a {difficulty} learner.)"
    try:
        result = extract_json(chat_once(model_choice, system_prompt, user_prompt))
        score = int(result.get("score", 0))
        notes = str(result.get("notes", "")).strip()
        assessed_difficulty = str(result.get("assessed_difficulty", "")).strip()
        return score, notes, assessed_difficulty
    except Exception as e:
        return 0, f"Could not parse judge output ({e})", ""

## Putting it together

One generator function drives the whole pipeline: generate all the questions up front, then for each one generate an answer and get it judged, yielding progress after every row so the Gradio UI can update live instead of freezing until everything is done.

In [8]:
def generate_dataset(topic, difficulty, count, q_model, a_model, judge_model, min_score, progress=gr.Progress()):
    if not topic.strip():
        raise gr.Error("Please enter a topic")

    rows = []
    yield pd.DataFrame(rows), f"Generating {count} questions about '{topic}'...", None

    questions = generate_questions(q_model, topic, difficulty, count)

    for i, question in enumerate(questions):
        progress((i + 1) / len(questions), desc=f"Answering & judging {i + 1}/{len(questions)}")
        answer = generate_answer(a_model, question, difficulty)
        score, notes, assessed_difficulty = judge_pair(judge_model, question, answer, difficulty)
        rows.append({
            "topic": topic,
            "difficulty": difficulty,
            "question": question,
            "question_model": q_model,
            "answer": answer,
            "answer_model": a_model,
            "judge_score": score,
            "judge_notes": notes,
            "judge_model": judge_model,
            "assessed_difficulty": assessed_difficulty,
            "difficulty_match": assessed_difficulty.lower() == difficulty.lower(),
            "kept": score >= min_score
        })
        yield pd.DataFrame(rows), f"Generated {i + 1}/{len(questions)} pairs...", None

    df = pd.DataFrame(rows)
    kept_df = df[df["kept"]]

    fd, path = tempfile.mkstemp(suffix=".jsonl")
    with os.fdopen(fd, "w") as f:
        for _, row in kept_df.iterrows():
            f.write(json.dumps({
                "question": row["question"],
                "answer": row["answer"],
                "topic": row["topic"],
                "difficulty": row["difficulty"],
            }) + "\n")

    mismatches = (~df["difficulty_match"]).sum()
    yield df, f"Done: {len(kept_df)}/{len(df)} pairs kept (score >= {min_score}). {mismatches} difficulty mismatch(es).", path

## The Gradio UI

In [ ]:
default_models = list(MODELS.keys())

with gr.Blocks(title="Synthetic Q&A Dataset Generator") as ui:
    gr.Markdown("# 🧪 Synthetic Q&A Dataset Generator")
    gr.Markdown(
        "Generate a quality-controlled technical Q&A dataset using three models in three different "
        "roles: one writes questions, one writes answers, one judges the pairs."
    )

    with gr.Row():
        topic = gr.Textbox(label="Topic", placeholder="e.g. Python generators, RAG, Docker networking", scale=2)
        difficulty = gr.Dropdown(["Beginner", "Intermediate", "Advanced"], value="Intermediate", label="Difficulty")
        count = gr.Slider(3, 20, value=5, step=1, label="Number of Q&A pairs")

    with gr.Row():
        q_model = gr.Dropdown(default_models, value=default_models[0], label="Question model")
        a_model = gr.Dropdown(default_models, value=default_models[min(1, len(default_models) - 1)], label="Answer model")
        judge_model = gr.Dropdown(default_models, value=default_models[0], label="Judge model")

    min_score = gr.Slider(1, 5, value=3, step=1, label="Minimum judge score to keep")

    generate_btn = gr.Button("Generate dataset", variant="primary")
    status = gr.Markdown("")
    table = gr.Dataframe(
        headers=["topic", "difficulty", "question", "question_model", "answer", "answer_model",
                 "judge_score", "judge_notes", "judge_model", "assessed_difficulty", "difficulty_match", "kept"],
        column_widths=["100px", "90px", "280px", "110px", "320px", "110px",
                       "70px", "200px", "110px", "120px", "100px", "70px"],
        wrap=True,
        max_height=600
    )
    download = gr.File(label="Download kept pairs (JSONL)")

    generate_btn.click(
        generate_dataset,
        inputs=[topic, difficulty, count, q_model, a_model, judge_model, min_score],
        outputs=[table, status, download]
    )

ui.launch()

## Where to take this further

- Add a fourth role: a model that rewrites low-scoring pairs instead of just discarding them
- Feed the kept JSONL straight into a fine-tuning run for the Week 1/2 technical tutor
- Track cost/latency per model to compare providers on this task, not just answer quality